## LCEL 인터페이스


사용자 정의 체인을 가능한 쉽게 만들 수 있도록, [`Runnable`](https://api.python.langchain.com/en/stable/runnables/langchain_core.runnables.base.Runnable.html#langchain_core.runnables.base.Runnable) 프로토콜을 구현했습니다. 

`Runnable` 프로토콜은 대부분의 컴포넌트에 구현되어 있습니다.

이는 표준 인터페이스로, 사용자 정의 체인을 정의하고 표준 방식으로 호출하는 것을 쉽게 만듭니다.
표준 인터페이스에는 다음이 포함됩니다.

- [`stream`](#stream): 응답의 청크를 스트리밍합니다.
- [`invoke`](#invoke): 입력에 대해 체인을 호출합니다.
- [`batch`](#batch): 입력 목록에 대해 체인을 호출합니다.

비동기 메소드도 있습니다.

- [`astream`](#async-stream): 비동기적으로 응답의 청크를 스트리밍합니다.
- [`ainvoke`](#async-invoke): 비동기적으로 입력에 대해 체인을 호출합니다.
- [`abatch`](#async-batch): 비동기적으로 입력 목록에 대해 체인을 호출합니다.
- [`astream_log`](#async-stream-intermediate-steps): 최종 응답뿐만 아니라 발생하는 중간 단계를 스트리밍합니다.

In [1]:
# API KEY를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API KEY 정보로드
load_dotenv()

True

In [2]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install -qU langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH01-Basic")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH01-Basic


LCEL 문법을 사용하여 chain 을 생성합니다.

In [6]:
# from langchain_openai import ChatOpenAI
from langchain_upstage import ChatUpstage
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ChatOpenAI 모델을 인스턴스화합니다.
# model = ChatOpenAI()
model = ChatUpstage()
# 주어진 토픽에 대한 농담을 요청하는 프롬프트 템플릿을 생성합니다.
prompt = PromptTemplate.from_template("{topic} 에 대하여 3문장으로 설명해줘.")
# 프롬프트와 모델을 연결하여 대화 체인을 생성합니다.
chain = prompt | model | StrOutputParser()

## stream: 실시간 출력


이 함수는 `chain.stream` 메서드를 사용하여 주어진 토픽에 대한 데이터 스트림을 생성하고, 이 스트림을 반복하여 각 데이터의 내용(`content`)을 즉시 출력합니다. `end=""` 인자는 출력 후 줄바꿈을 하지 않도록 설정하며, `flush=True` 인자는 출력 버퍼를 즉시 비우도록 합니다. 

In [7]:
# chain.stream 메서드를 사용하여 '멀티모달' 토픽에 대한 스트림을 생성하고 반복합니다.
for token in chain.stream({"topic": "멀티모달"}):
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

멀티모달은 인공지능 시스템이 텍스트, 이미지, 음성 등 다양한 형태의 입력을 동시에 처리하고 이해할 수 있도록 하는 기술입니다. 이를 통해 더 풍부한 정보를 활용하여 더 정확한 결과나 서비스를 제공할 수 있습니다. 예를 들어, 멀티모달 인공지능은 이미지 속 사물을 인식하고 이에 대한 질문을 자연어로 이해하고 답할 수 있습니다.

In [8]:
answer = chain.stream({"topic": "멀티모달"})

In [9]:
for token in answer:
    print(token, end="", flush=True)

멀티모달은 인공지능이 텍스트, 이미지, 음성 등 다양한 형태의 데이터를 동시에 처리하고 이해할 수 있도록 하는 기술입니다. 이 기술은 각각의 모달리티를 독립적으로 처리하는 것이 아니라, 상호 연관성을 고려하여 더욱 정교한 인식과 생성을 가능하게 합니다. 예를 들어, 멀티모달 AI는 사진 속 사물을 인식하고 그에 대한 설명을 텍스트로 생성하거나, 텍스트 명령을 받아 이미지를 생성하는 등 복합적인 작업을 수행할 수 있습니다.

## invoke: 호출


`chain` 객체의 `invoke` 메서드는 주제를 인자로 받아 해당 주제에 대한 처리를 수행합니다.

In [10]:
# chain 객체의 invoke 메서드를 호출하고, 'ChatGPT'라는 주제로 딕셔너리를 전달합니다.
chain.invoke({"topic": "ChatGPT"})

'ChatGPT는 San Francisco에 기반을 둔 AI 회사 Upstage에서 개발한, 107억 매개변수를 가진 대규모 언어 모델(LLM)입니다. 이 모델은 대화형 AI에 특화되어 사용자와 자연스럽고 매력적인 대화를 제공할 수 있으며, 다양한 주제에 대한 질문에 답변하고, 문제 해결을 돕는 등 다재다능한 활용성을 갖추고 있습니다.'

## batch: 배치(단위 실행)


함수 `chain.batch`는 여러 개의 딕셔너리를 포함하는 리스트를 인자로 받아, 각 딕셔너리에 있는 `topic` 키의 값을 사용하여 일괄 처리를 수행합니다.

In [12]:
# 주어진 토픽 리스트를 batch 처리하는 함수 호출
answer = chain.batch([{"topic": "ChatGPT"}, {"topic": "Instagram"}])

In [13]:
answer[0]

'ChatGPT는 AI 스타트업인 업스테이지가 개발한, 107억 매개변수를 가진 언어모델입니다. 이는 대화형 AI 모델로, 사용자와의 상호작용을 통해 자연스러운 대화를 수행하고 다양한 질문에 답변할 수 있습니다. ChatGPT는 풍부한 지식과 상식을 바탕으로 정확한 정보를 제공하고, 창의적인 아이디어를 제안하며, 프로그래밍 코드 작성도 도와주는 등 다양한 기능을 제공합니다.'

In [14]:
answer[1]

'인스타그램은 2010년 10월에 출시된 사진 및 동영상 공유 기반의 소셜 미디어 플랫폼입니다. 사용자들은 자신의 일상을 사진이나 동영상으로 촬영하여 공유하고, 해시태그나 지인 태그 기능을 통해 다른 사용자들과 소통할 수 있습니다. 또한, 인스타그램은 스토리, IGTV, 릴스 등 다양한 기능을 제공하여 사용자들의 창의적인 표현을 지원하고, 비즈니스 계정 기능을 통해 마케팅 플랫폼으로도 활용되고 있습니다.'

`max_concurrency` 매개변수를 사용하여 동시 요청 수를 설정할 수 있습니다

`config` 딕셔너리는 `max_concurrency` 키를 통해 동시에 처리할 수 있는 최대 작업 수를 설정합니다. 여기서는 최대 3개의 작업을 동시에 처리하도록 설정되어 있습니다.

In [15]:
chain.batch(
    [
        {"topic": "ChatGPT"},
        {"topic": "Instagram"},
        {"topic": "멀티모달"},
        {"topic": "프로그래밍"},
        {"topic": "머신러닝"},
    ],
    config={"max_concurrency": 3},
)

['ChatGPT는 San Francisco에 기반을 둔 AI 기업인 OpenAI에서 개발한 대화형 AI 챗봇입니다. 1750억 개의 매개변수를 가진 대규모 언어 모델으로, 다양한 주제에 대해 인간의 대화체와 유사하게 소통할 수 있습니다. ChatGPT는 주어진 프롬프트에 대해 정확하고 적절하며 재미있는 응답을 생성할 수 있어, 사용자와의 상호작용에서 높은 몰입감과 만족도를 제공합니다.',
 '인스타그램은 사진 및 동영상 공유를 중심으로 한 소셜 미디어 플랫폼으로, 사용자들은 다양한 필터를 적용하여 창의적인 콘텐츠를 제작하고 팔로워들과 공유할 수 있습니다. 또한, 인스타그램은 스토리와 릴스 등의 기능을 통해 사용자들이 일시적인 콘텐츠나 숏폼 동영상을 공유하며 더 역동적이고 즉각적인 소통을 가능하게 합니다. 페이스북이 소유한 이 플랫폼은 전 세계적으로 활발한 사용자 기반을 보유하고 있으며, 개인 사용자들뿐만 아니라 기업들과 창작자들 또한 마케팅과 커뮤니티 구축에 활용하고 있습니다.',
 '멀티모달(Multimodal)은 인공지능 분야에서 여러 입력 채널을 결합하여 더 풍부한 정보를 처리하고 이해하는 방식을 의미합니다. 예를 들어, 텍스트, 이미지, 음성 등 다양한 형태의 데이터를 동시에 활용하여 더 정확한 결과를 도출하거나, 사용자의 의도를 더욱 정확하게 파악할 수 있습니다. 이를 통해 멀티모달 AI는 더욱 자연스럽고 효과적인 인간-컴퓨터 상호작용을 가능하게 합니다.',
 '프로그래밍은 소프트웨어 개발을 위한 일련의 절차와 방법을 의미하며, 사람이 이해할 수 있는 자연어와는 달리 컴퓨터가 이해할 수 있는 구조화된 언어를 사용해 프로그램을 작성하는 작업을 말합니다. 이는 문제 해결을 위한 논리적 사고와 알고리즘 설계, 코드 작성, 테스트, 디버깅, 유지보수를 포함하는 광범위한 과정이며, 현대 사회에서 디지털 기술 활용의 핵심 역할을 합니다.',
 '머신러닝은 인공지능의 한 분야로, 컴퓨터가 데이터를 통해 학습하고 경험으로 개선되도록 만드는 알고리즘과 모델을 연구하

## async stream: 비동기 스트림


함수 `chain.astream`은 비동기 스트림을 생성하며, 주어진 토픽에 대한 메시지를 비동기적으로 처리합니다.

비동기 for 루프(`async for`)를 사용하여 스트림에서 메시지를 순차적으로 받아오고, `print` 함수를 통해 메시지의 내용(`s.content`)을 즉시 출력합니다. `end=""`는 출력 후 줄바꿈을 하지 않도록 설정하며, `flush=True`는 출력 버퍼를 강제로 비워 즉시 출력되도록 합니다.


In [16]:
# 비동기 스트림을 사용하여 'YouTube' 토픽의 메시지를 처리합니다.
async for token in chain.astream({"topic": "YouTube"}):
    # 메시지 내용을 출력합니다. 줄바꿈 없이 바로 출력하고 버퍼를 비웁니다.
    print(token, end="", flush=True)

YouTube는 구글이 소유한 동영상 공유 플랫폼으로, 사용자가 동영상을 업로드, 공유, 시청할 수 있는 웹사이트입니다. 월간 활성 사용자가 20억 명이 넘는 세계에서 두 번째로 많이 방문하는 웹사이트이며, 엔터테인먼트, 교육, 소셜 미디어 등 다양한 목적으로 이용됩니다. YouTube는 개인 창작자에게 콘텐츠 배포의 장을 제공하고, 사용자는 다양한 주제의 영상들을 통해 정보 습득과 여가를 즐깁니다.

## async invoke: 비동기 호출


`chain` 객체의 `ainvoke` 메서드는 비동기적으로 주어진 인자를 사용하여 작업을 수행합니다. 여기서는 `topic`이라는 키와 `NVDA`(엔비디아의 티커) 라는 값을 가진 딕셔너리를 인자로 전달하고 있습니다. 이 메서드는 특정 토픽에 대한 처리를 비동기적으로 요청하는 데 사용될 수 있습니다.


In [17]:
# 비동기 체인 객체의 'ainvoke' 메서드를 호출하여 'NVDA' 토픽을 처리합니다.
my_process = chain.ainvoke({"topic": "NVDA"})

In [18]:
# 비동기로 처리되는 프로세스가 완료될 때까지 기다립니다.
await my_process

'NVDA는 NVIDIA의 티커 심볼로, 인공지능(AI) 및 그래픽 처리 기술 분야에서 세계적인 선두주자입니다. 이 기업은 게이밍, 전문 시각화, 데이터 센터, 자동차 등 다양한 시장에서 활약하며, AI와 머신러닝 기술의 발전으로 인해 큰 성장 기회를 맞이하고 있습니다.'

## async batch: 비동기 배치


함수 `abatch`는 비동기적으로 일련의 작업을 일괄 처리합니다.

이 예시에서는 `chain` 객체의 `abatch` 메서드를 사용하여 `topic` 에 대한 작업을 비동기적으로 처리하고 있습니다.

`await` 키워드는 해당 비동기 작업이 완료될 때까지 기다리는 데 사용됩니다.


In [19]:
# 주어진 토픽에 대해 비동기적으로 일괄 처리를 수행합니다.
my_abatch_process = chain.abatch(
    [{"topic": "YouTube"}, {"topic": "Instagram"}, {"topic": "Facebook"}]
)

In [20]:
# 비동기로 처리되는 일괄 처리 프로세스가 완료될 때까지 기다립니다.
await my_abatch_process

['YouTube는 구글이 소유한 동영상 공유 플랫폼으로, 사용자가 동영상을 업로드, 공유, 시청할 수 있는 공간을 제공합니다. 월간 활성 사용자 수가 20억 명 이상인 이 플랫폼은 교육, 엔터테인먼트, 소셜 미디어 등 다양한 목적으로 널리 사용되며, 많은 콘텐츠 크리에이터와 인플루언서들이 자신의 채널을 운영하며 수익을 창출하는 곳이기도 합니다.',
 '인스타그램은 사진 및 동영상 공유를 중심으로 하는 소셜 미디어 플랫폼입니다. 사용자는 자신의 창의적인 콘텐츠를 공유하고, 해시태그나 지인 검색을 통해 전 세계의 다양한 게시물을 탐색할 수 있습니다. 또한, 인스타그램은 스토리, IGTV, 릴스 등 다양한 기능을 제공하여 사용자들이 더욱 재미있고 다채롭게 소통할 수 있도록 돕습니다.',
 'Facebook은 2004년 마크 주커버그가 설립한 세계적인 소셜 미디어 플랫폼으로, 사용자들이 친구 및 가족과 연결하고 정보를 공유하며 커뮤니티를 형성할 수 있는 공간을 제공합니다. 월간 활성 사용자가 20억 명이 넘는 Facebook은 뉴스피드, 메신저, 그룹, 마켓플레이스 등의 다양한 기능을 통해 사용자들의 소통과 정보 교류를 지원하고 있습니다. 또한, Facebook은 오큘러스 VR과 같은 기술을 통해 가상 현실 분야로도 사업을 확장하고 있습니다.']

## Parallel: 병렬성

LangChain Expression Language가 병렬 요청을 지원하는 방법을 살펴봅시다.
예를 들어, `RunnableParallel`을 사용할 때, 각 요소를 병렬로 실행합니다.


`langchain_core.runnables` 모듈의 `RunnableParallel` 클래스를 사용하여 두 가지 작업을 병렬로 실행하는 예시를 보여줍니다.

`ChatPromptTemplate.from_template` 메서드를 사용하여 주어진 `country`에 대한 **수도** 와 **면적** 을 구하는 두 개의 체인(`chain1`, `chain2`)을 만듭니다.

이 체인들은 각각 `model`과 파이프(`|`) 연산자를 통해 연결됩니다. 마지막으로, `RunnableParallel` 클래스를 사용하여 이 두 체인을 `capital`와 `area`이라는 키로 결합하여 동시에 실행할 수 있는 `combined` 객체를 생성합니다.


In [21]:
from langchain_core.runnables import RunnableParallel

# {country} 의 수도를 물어보는 체인을 생성합니다.
chain1 = (
    PromptTemplate.from_template("{country} 의 수도는 어디야?")
    | model
    | StrOutputParser()
)

# {country} 의 면적을 물어보는 체인을 생성합니다.
chain2 = (
    PromptTemplate.from_template("{country} 의 면적은 얼마야?")
    | model
    | StrOutputParser()
)

# 위의 2개 체인을 동시에 생성하는 병렬 실행 체인을 생성합니다.
combined = RunnableParallel(capital=chain1, area=chain2)

`chain1.invoke()` 함수는 `chain1` 객체의 `invoke` 메서드를 호출합니다.

이때, `country`이라는 키에 `대한민국`라는 값을 가진 딕셔너리를 인자로 전달합니다.


In [22]:
# chain1 를 실행합니다.
chain1.invoke({"country": "대한민국"})

'대한민국의 수도는 서울입니다. 서울은 대한민국의 최대 도시이자 정치, 경제, 문화의 중심지입니다.'

이번에는 `chain2.invoke()` 를 호출합니다. `country` 키에 다른 국가인 `미국` 을 전달합니다.


In [ ]:
# chain2 를 실행합니다.
chain2.invoke({"country": "미국"})

`combined` 객체의 `invoke` 메서드는 주어진 `country`에 대한 처리를 수행합니다.

이 예제에서는 `대한민국`라는 주제를 `invoke` 메서드에 전달하여 실행합니다.


In [ ]:
# 병렬 실행 체인을 실행합니다.
combined.invoke({"country": "대한민국"})

### 배치에서의 병렬 처리

병렬 처리는 다른 실행 가능한 코드와 결합될 수 있습니다.
배치와 병렬 처리를 사용해 보도록 합시다.


`chain1.batch` 함수는 여러 개의 딕셔너리를 포함하는 리스트를 인자로 받아, 각 딕셔너리에 있는 "topic" 키에 해당하는 값을 처리합니다. 이 예시에서는 "대한민국"와 "미국"라는 두 개의 토픽을 배치 처리하고 있습니다.


In [ ]:
# 배치 처리를 수행합니다.
chain1.batch([{"country": "대한민국"}, {"country": "미국"}])

`chain2.batch` 함수는 여러 개의 딕셔너리를 리스트 형태로 받아, 일괄 처리(batch)를 수행합니다.

이 예시에서는 `대한민국`와 `미국`라는 두 가지 국가에 대한 처리를 요청합니다.


In [ ]:
# 배치 처리를 수행합니다.
chain2.batch([{"country": "대한민국"}, {"country": "미국"}])

`combined.batch` 함수는 주어진 데이터를 배치로 처리하는 데 사용됩니다. 이 예시에서는 두 개의 딕셔너리 객체를 포함하는 리스트를 인자로 받아 각각 `대한민국`와 `미국` 두 나라에 대한 데이터를 배치 처리합니다.


In [ ]:
# 주어진 데이터를 배치로 처리합니다.
combined.batch([{"country": "대한민국"}, {"country": "미국"}])